# ARF-ADWIN (Adaptive Random Forest + ADWIN) Training Notebook

This notebook trains an **Online Learning** ARF-ADWIN model on the preprocessed CICEVSE2024 dataset using the `river` library. Unlike the static models (SVM, RF, DT, LR), this model learns incrementally in a **prequential (predict-then-learn)** fashion and can detect **concept drift** in real-time.

**Dataset**: CICEVSE2024 Network Traffic (14 attack classes + benign)  
**Algorithm**: Adaptive Random Forest with ADWIN drift detection (`river.forest.ARFClassifier`)  
**Task**: Multiclass classification of EV charging network intrusions  
**Training Strategy**: Prequential (instance-by-instance predict → evaluate → learn)

In [ ]:
import os
import json
import pickle
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from river import forest, preprocessing, drift, metrics, stream
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Plotting config
%matplotlib inline
plt.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid', palette='deep')

## 1. Load Data
We load a sample for visualisation and stream the full training data during the prequential loop.

In [ ]:
# Adjust path
if os.path.exists('../../../data/processed'):
    DATA_DIR = '../../../data/processed'
elif os.path.exists('data/processed'):
    DATA_DIR = 'data/processed'
else:
    DATA_DIR = '../data/processed'

print(f"Using data directory: {DATA_DIR}")

# Load targets and a sample for visualisation
y_train = pd.read_csv(os.path.join(DATA_DIR, "y_train.csv"))
y_test = pd.read_csv(os.path.join(DATA_DIR, "y_test.csv"))
X_train_sample = pd.read_csv(os.path.join(DATA_DIR, "X_train.csv"), nrows=50000)

print(f"y_train shape: {y_train.shape}")
print(f"X_train_sample shape: {X_train_sample.shape}")
print(f"Number of features: {X_train_sample.shape[1]}")

## 1.1 Dataset Overview

In [ ]:
total_train = len(y_train)
X_test_tmp = pd.read_csv(os.path.join(DATA_DIR, "X_test.csv"), nrows=1)
X_val_tmp = pd.read_csv(os.path.join(DATA_DIR, "X_val.csv"), nrows=1)
# Get actual row counts from y files
y_val = pd.read_csv(os.path.join(DATA_DIR, "y_val.csv"))

print(f"Full X_train rows : {total_train:,}")
print(f"X_val rows        : {len(y_val):,}")
print(f"X_test rows       : {len(y_test):,}")
print(f"\nNumber of features: {X_train_sample.shape[1]}")
print(f"\nData types:\n{X_train_sample.dtypes.value_counts()}")
print(f"\nMissing values: {X_train_sample.isnull().sum().sum()} total")

print("\nSummary Statistics (first 10 features):")
X_train_sample.iloc[:, :10].describe().round(3)

## 1.2 Train / Validation / Test Split Sizes

In [ ]:
split_sizes = pd.DataFrame({
    'Split': ['Train', 'Validation', 'Test'],
    'Samples': [total_train, len(y_val), len(y_test)]
})
split_sizes['Percentage'] = (split_sizes['Samples'] / split_sizes['Samples'].sum() * 100).round(1)

fig, ax = plt.subplots(figsize=(8, 3))
bars = ax.barh(split_sizes['Split'], split_sizes['Samples'], color=['#2196F3', '#FF9800', '#4CAF50'])
for bar, pct in zip(bars, split_sizes['Percentage']):
    ax.text(bar.get_width() + 5000, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():,.0f} ({pct}%)', va='center', fontsize=11)
ax.set_xlabel('Number of Samples')
ax.set_title('Train / Validation / Test Split Sizes')
plt.tight_layout()
plt.show()

## 1.3 Visualize Class Distributions

In [ ]:
multi_counts = y_train['Label_Multiclass'].value_counts()
colors = sns.color_palette('viridis', len(multi_counts))

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(multi_counts.index, multi_counts.values, color=colors)
ax.set_title('Multiclass Distribution (Train)', fontsize=14)
ax.set_xlabel('Count')
ax.set_ylabel('Attack Type')
for i, val in enumerate(multi_counts.values):
    ax.text(val + 1000, i, f'{val:,}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\nMulticlass label counts:")
print(multi_counts.to_string())

## 1.4 Feature Correlation Heatmap

In [ ]:
top_features = X_train_sample.var().nlargest(30).index.tolist()
corr_matrix = X_train_sample[top_features].corr()

plt.figure(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, cmap='coolwarm', center=0,
            square=True, linewidths=0.5, fmt='.1f',
            cbar_kws={'shrink': 0.8, 'label': 'Pearson Correlation'})
plt.title('Feature Correlation Heatmap (Top 30 by Variance)', fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

## 1.5 Feature Distribution Box Plots

In [ ]:
top10 = X_train_sample.var().nlargest(10).index.tolist()

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, col in enumerate(top10):
    sample = X_train_sample[col].sample(n=min(10000, len(X_train_sample)), random_state=42)
    axes[i].boxplot(sample.values, vert=True, patch_artist=True,
                    boxprops=dict(facecolor='#26A69A', alpha=0.7))
    axes[i].set_title(col, fontsize=9, fontweight='bold')
    axes[i].tick_params(axis='x', labelbottom=False)

plt.suptitle('Top 10 Features by Variance — Box Plots (Scaled Data)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 2. Initialize ARF-ADWIN Model

We initialize the ARF+ADWIN pipeline following the Makhmudov et al. (2025) parameters:
- `n_models=20` (20 trees in the forest)
- `max_features=0.5` (50% feature subsampling)
- `grace_period=30` (samples before first split)
- `ADWIN(delta=0.002)` for concept drift detection

In [ ]:
# Initialize Pipeline
scaler = preprocessing.StandardScaler()

arf = forest.ARFClassifier(
    n_models=20,
    max_features=0.5,
    grace_period=30,
    leaf_prediction='nba',  # Naive Bayes Adaptive
    drift_detector=drift.ADWIN(delta=0.002),
    warning_detector=drift.ADWIN(delta=0.002)
)

model_pipeline = scaler | arf

# Standalone drift detector for logging
drift_detector_log = drift.ADWIN(delta=0.002)

# Tracking
accuracy_tracker = metrics.Accuracy()
drift_events = []
y_true_list = []
y_pred_list = []
accuracy_over_time = []

print("ARF-ADWIN pipeline initialized.")
print(f"  Trees: {arf.n_models}")
print(f"  Max features: {arf.max_features}")
print(f"  Drift detector: ADWIN(delta=0.002)")

## 3. Prequential Training (Predict → Evaluate → Learn)

⚠️ **Note**: This cell processes data row-by-row which is computationally intensive. Set `MAX_ROWS` below to control training duration. The paper uses the full dataset, but for demonstration a subset can be used.

To run on the **full dataset**, set `MAX_ROWS = None`.

In [ ]:
# Set to None for full dataset, or a number for quick demo
MAX_ROWS = 500000  # Change to None for full training

X_train_path = os.path.join(DATA_DIR, "X_train.csv")
y_train_path = os.path.join(DATA_DIR, "y_train.csv")

chunksize = 10000
row_count = 0
max_rows = MAX_ROWS if MAX_ROWS is not None else float('inf')
start_time = time.time()

print(f"Starting Prequential Training (max rows: {MAX_ROWS or 'ALL'})...")

x_iter = pd.read_csv(X_train_path, chunksize=chunksize)
y_iter = pd.read_csv(y_train_path, chunksize=chunksize)

try:
    for chunk_idx, (X_chunk, y_chunk) in enumerate(zip(x_iter, y_iter)):
        if row_count >= max_rows:
            break
        
        if row_count + len(X_chunk) > max_rows:
            X_chunk = X_chunk.iloc[:int(max_rows - row_count)]
            y_chunk = y_chunk.iloc[:int(max_rows - row_count)]
        
        # Use Label_Multiclass for multiclass classification
        if 'Label_Multiclass' in y_chunk.columns:
            y_series = y_chunk['Label_Multiclass']
        else:
            y_series = y_chunk.iloc[:, 0]
        
        x_dict_list = X_chunk.to_dict(orient='records')
        y_list = y_series.tolist()
        
        for x_dict, y_val in zip(x_dict_list, y_list):
            # 1. Predict
            y_pred = model_pipeline.predict_one(x_dict)
            
            y_true_list.append(y_val)
            y_pred_list.append(y_pred if y_pred is not None else y_val)
            
            if y_pred is not None:
                # 2. Update metric
                accuracy_tracker.update(y_val, y_pred)
                
                # 3. Check for drift
                is_correct = 1.0 if y_val == y_pred else 0.0
                drift_detector_log.update(is_correct)
                
                if drift_detector_log.drift_detected:
                    drift_events.append({
                        'instance': row_count,
                        'accuracy_at_drift': accuracy_tracker.get()
                    })
            
            # 4. Learn
            model_pipeline.learn_one(x_dict, y_val)
            
            row_count += 1
        
        # Record accuracy every chunk
        accuracy_over_time.append({
            'instances': row_count,
            'accuracy': accuracy_tracker.get()
        })
        
        elapsed = time.time() - start_time
        if (chunk_idx + 1) % 5 == 0:
            print(f"  Chunk {chunk_idx+1}: {row_count:,} rows | Accuracy: {accuracy_tracker.get():.4f} | {elapsed:.1f}s")

except StopIteration:
    pass

total_time = time.time() - start_time
print(f"\nTraining complete!")
print(f"  Total rows processed: {row_count:,}")
print(f"  Final accuracy: {accuracy_tracker.get():.4f}")
print(f"  Drift events detected: {len(drift_events)}")
print(f"  Total time: {total_time:.1f}s")

## 4. Accuracy Over Time
One of the key advantages of online learning is being able to track model performance as it learns.

In [ ]:
if accuracy_over_time:
    acc_df = pd.DataFrame(accuracy_over_time)
    
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(acc_df['instances'], acc_df['accuracy'], color='#1976D2', linewidth=2, label='Rolling Accuracy')
    
    # Mark drift events
    if drift_events:
        drift_df = pd.DataFrame(drift_events)
        ax.scatter(drift_df['instance'], drift_df['accuracy_at_drift'],
                   color='red', marker='x', s=100, zorder=5, label=f'Drift Events ({len(drift_events)})')
    
    ax.set_xlabel('Instances Processed', fontsize=12)
    ax.set_ylabel('Accuracy', fontsize=12)
    ax.set_title('ARF-ADWIN: Accuracy Over Time (Prequential)', fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No accuracy data recorded.")

## 5. Evaluation
Evaluate the prequential predictions using standard sklearn metrics.

In [ ]:
print(f"Total predictions: {len(y_true_list):,}")
print(f"\nOverall Accuracy: {accuracy_score(y_true_list, y_pred_list):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_true_list, y_pred_list, zero_division=0))

# Confusion Matrix
unique_labels = sorted(set(y_true_list) | set(y_pred_list))
cm = confusion_matrix(y_true_list, y_pred_list, labels=unique_labels)

fig_size = max(8, len(unique_labels) * 0.8)
plt.figure(figsize=(fig_size + 2, fig_size))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=unique_labels, yticklabels=unique_labels)
plt.title('ARF-ADWIN Prequential Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

## 6. Drift Events Log

In [ ]:
if drift_events:
    drift_df = pd.DataFrame(drift_events)
    print(f"Total drift events: {len(drift_events)}")
    print(drift_df.to_string(index=False))
else:
    print("No concept drift events were detected during training.")

## 7. Save Model and Predictions

In [ ]:
if os.path.exists('../../../saved_models'):
    SAVE_DIR = '../../../saved_models'
    PREDS_DIR = '../../../predictions'
elif os.path.exists('saved_models'):
    SAVE_DIR = 'saved_models'
    PREDS_DIR = 'predictions'
else:
    SAVE_DIR = '../saved_models'
    PREDS_DIR = '../predictions'

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(PREDS_DIR, exist_ok=True)

# Save model
model_path = os.path.join(SAVE_DIR, 'arf_adwin.pkl')
with open(model_path, 'wb') as f:
    pickle.dump(model_pipeline, f)
print(f"Model saved to {model_path}")

# Save predictions
pred_df = pd.DataFrame({
    'y_true': y_true_list,
    'y_pred': y_pred_list
})
pred_path = os.path.join(PREDS_DIR, 'arfadwin_preds_multiclass.csv')
pred_df.to_csv(pred_path, index=False)
print(f"Predictions saved to {pred_path}")

# Save drift events
drift_path = os.path.join(PREDS_DIR, 'arf_adwin_drift_events.json')
with open(drift_path, 'w') as f:
    json.dump(drift_events, f, indent=4)
print(f"Drift events saved to {drift_path}")

print("\nARF-ADWIN pipeline complete!")